# N01 · Fábrica de datos sintéticos

## Pregunta

> ¿Puedo generar problemas donde **conozca la respuesta exacta**, y que cubran todo lo que
> los treinta notebooks siguientes van a necesitar?

## Hipótesis

*(Escríbela antes de ejecutar nada, y no la edites después.)*

Creo que la parte difícil no será generar los datos —eso es NumPy— sino calcular el
**techo teórico** de cada uno. Sin ese número, un resultado no se puede interpretar.

Espero que haya generadores donde el techo salga de una fórmula cerrada y otros donde
solo pueda estimarlo. Reconocer la diferencia es parte del ejercicio.

---

## Por qué este notebook existe

Después del N00 tengo un arnés que sabe entrenar cualquier cosa. Lo que no tengo es
**qué** entrenar.

Este notebook construye el único dataset que atraviesa el itinerario entero. Se define
aquí y no se vuelve a tocar: los otros veintinueve notebooks solo lo importan.

Dos partes, deliberadamente separadas:

| Parte | Qué | Para |
|---|---|---|
| **A · `lab/data.py`** | Tabular y 2D | N02 – N17 |
| **B · `lab/arithmetic.py`** | El lenguaje aritmético | N18 – N30 |

## Las cuatro decisiones de diseño

### 1. Cada dataset trae su techo

Es la decisión que define el módulo. Un dataset no es solo `(X, y)`: es `(X, y, cuál es
el mejor resultado posible)`.

```python
dataset.ceiling         # 0.25
dataset.ceiling_metric  # "mse"
```

Sin ese número, un 0.87 no significa nada: no sé si me falta un 1 % o un 12 %. **Con él,
cada experimento tiene una referencia.**

### 2. Todo lo que querré romper después es un parámetro

Ruido de etiqueta, desbalanceo, duplicados, estructura de grupo. Si en N10 o N15 tengo
que reescribir el generador para provocar un fallo, el generador está mal diseñado.

### 3. Los generadores se registran en el arnés, no lo modifican

`@harness.datasets.register("spirals")`. N01 **no toca `harness.py`**. Esa es la prueba
de que el registro de N00 estaba bien planteado.

### 4. El lenguaje aritmético es aritmética, y nada más

Sin gramáticas inventadas ni semánticas propias. Diecisiete tokens, verificador de tres
líneas. Si tengo que explicar el dataset, está mal elegido.

In [ ]:
# ── ARRANQUE ── (igual en todos los notebooks)
import os, sys
from pathlib import Path

while not (Path.cwd() / "lab").exists() and Path.cwd() != Path.cwd().parent:
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))

print("raíz del proyecto:", Path.cwd())

---
---

# Parte A · Datos tabulares y 2D

Siete generadores y cinco modificadores. Todos con semilla, todos con techo.

In [ ]:
%%writefile lab/data.py
"""Synthetic tabular and 2D datasets, each carrying its own performance ceiling.

ES: Generadores sintéticos. Cada uno sabe cuál es el mejor resultado posible,
que es lo que permite distinguir un 0.87 excelente de un 0.87 mediocre.

Comment convention / Convenio: docstrings in English say WHAT, Spanish notes say WHY.
"""
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Literal

import numpy as np
import torch
from scipy.stats import norm
from torch.utils.data import DataLoader, TensorDataset

from lab import harness

Task = Literal["regression", "classification"]
SplitStrategy = Literal["random", "by_group", "temporal"]


# ─────────────────────────────────────────────────────────────────────────────
# The container
# ─────────────────────────────────────────────────────────────────────────────
@dataclass
class SyntheticDataset:
    """Inputs, targets, and the best score any model could possibly reach.

    ES: `ceiling` es la razón de ser de esta clase. Sin ese número, un
    resultado no se puede interpretar: no sabes si te falta un 1% o un 12%.
    """

    inputs: np.ndarray
    targets: np.ndarray
    task: Task
    ceiling: float
    ceiling_metric: str
    description: str
    groups: np.ndarray | None = None
    metadata: dict = field(default_factory=dict)

    def __post_init__(self) -> None:
        self.inputs = np.asarray(self.inputs, dtype=np.float32)
        dtype = np.float32 if self.task == "regression" else np.int64
        self.targets = np.asarray(self.targets, dtype=dtype)

    def __len__(self) -> int:
        return len(self.inputs)

    @property
    def n_features(self) -> int:
        return self.inputs.shape[1]

    @property
    def n_classes(self) -> int:
        return int(self.targets.max()) + 1 if self.task == "classification" else 0

    def summary(self) -> str:
        return (f"{self.description}\n"
                f"  samples={len(self)}  features={self.n_features}  task={self.task}\n"
                f"  ceiling: {self.ceiling:.4f} {self.ceiling_metric}")

    # ── splitting ────────────────────────────────────────────────────────────
    def split(self, val_fraction: float = 0.2, strategy: SplitStrategy = "random",
              seed: int = 0) -> tuple["SyntheticDataset", "SyntheticDataset"]:
        """Split into train and validation.

        ES: `by_group` y `temporal` existen porque el split aleatorio miente
        cuando los datos tienen estructura. Se demuestra en N15.
        """
        indices = self._split_indices(val_fraction, strategy, seed)
        return self._subset(indices["train"]), self._subset(indices["val"])

    def _split_indices(self, val_fraction: float, strategy: SplitStrategy,
                       seed: int) -> dict[str, np.ndarray]:
        n = len(self)
        rng = np.random.default_rng(seed)

        if strategy == "temporal":
            cut = int(n * (1 - val_fraction))
            return {"train": np.arange(cut), "val": np.arange(cut, n)}

        if strategy == "by_group":
            if self.groups is None:
                raise ValueError("this dataset has no groups; generate it with n_groups>1")
            unique = np.unique(self.groups)
            rng.shuffle(unique)
            n_val_groups = max(1, int(len(unique) * val_fraction))
            val_groups = set(unique[:n_val_groups])
            is_val = np.array([g in val_groups for g in self.groups])
            return {"train": np.where(~is_val)[0], "val": np.where(is_val)[0]}

        shuffled = rng.permutation(n)
        cut = int(n * (1 - val_fraction))
        return {"train": shuffled[:cut], "val": shuffled[cut:]}

    def _subset(self, indices: np.ndarray) -> "SyntheticDataset":
        return SyntheticDataset(
            inputs=self.inputs[indices], targets=self.targets[indices],
            task=self.task, ceiling=self.ceiling, ceiling_metric=self.ceiling_metric,
            description=self.description,
            groups=None if self.groups is None else self.groups[indices],
            metadata=self.metadata,
        )

    # ── torch ────────────────────────────────────────────────────────────────
    def to_loader(self, batch_size: int = 32, shuffle: bool = False) -> DataLoader:
        tensors = TensorDataset(torch.from_numpy(self.inputs), torch.from_numpy(self.targets))
        return DataLoader(tensors, batch_size=batch_size, shuffle=shuffle)

    def to_loaders(self, batch_size: int = 32, val_fraction: float = 0.2,
                   strategy: SplitStrategy = "random",
                   seed: int = 0) -> tuple[DataLoader, DataLoader]:
        train, val = self.split(val_fraction, strategy, seed)
        return train.to_loader(batch_size, shuffle=True), val.to_loader(batch_size)


# ─────────────────────────────────────────────────────────────────────────────
# Generators
# ─────────────────────────────────────────────────────────────────────────────
def make_line(n_samples: int = 512, slope: float = 3.0, intercept: float = 2.0,
              noise_std: float = 0.5, seed: int = 0) -> SyntheticDataset:
    """y = slope*x + intercept + noise. Ceiling: MSE = noise_std**2."""
    rng = np.random.default_rng(seed)
    x = rng.uniform(-3, 3, size=(n_samples, 1))
    y = slope * x[:, 0] + intercept + rng.normal(0, noise_std, n_samples)
    return SyntheticDataset(
        inputs=x, targets=y, task="regression",
        ceiling=noise_std ** 2, ceiling_metric="mse",
        description=f"line y={slope}x+{intercept}, noise_std={noise_std}",
        metadata={"slope": slope, "intercept": intercept, "noise_std": noise_std},
    )


def make_two_gaussians(n_samples: int = 512, separation: float = 2.0,
                       spread: float = 1.0, n_features: int = 2,
                       seed: int = 0) -> SyntheticDataset:
    """Two isotropic gaussian blobs. Ceiling: the Bayes accuracy.

    ES: Con dos gaussianas de igual covarianza, el error de Bayes es exacto:
    Phi(-d / 2*sigma). Es el único caso del módulo donde el techo sale de una
    fórmula cerrada, y por eso es el mejor ejemplo para explicar el concepto.
    """
    rng = np.random.default_rng(seed)
    half = n_samples // 2
    offset = np.zeros(n_features)
    offset[0] = separation

    class_0 = rng.normal(0, spread, size=(half, n_features))
    class_1 = rng.normal(0, spread, size=(n_samples - half, n_features)) + offset
    inputs = np.vstack([class_0, class_1])
    targets = np.concatenate([np.zeros(half), np.ones(n_samples - half)])

    order = rng.permutation(n_samples)
    bayes_accuracy = 1 - norm.cdf(-separation / (2 * spread))

    return SyntheticDataset(
        inputs=inputs[order], targets=targets[order], task="classification",
        ceiling=bayes_accuracy, ceiling_metric="accuracy",
        description=f"two gaussians, separation={separation}, spread={spread}",
        metadata={"separation": separation, "spread": spread},
    )


def make_xor(n_samples: int = 512, noise_std: float = 0.1,
             seed: int = 0) -> SyntheticDataset:
    """The four XOR quadrants. Ceiling: perfect, and no line can reach it."""
    rng = np.random.default_rng(seed)
    corners = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
    labels = np.array([0, 1, 1, 0])
    picked = rng.integers(0, 4, size=n_samples)
    inputs = corners[picked] + rng.normal(0, noise_std, size=(n_samples, 2))
    return SyntheticDataset(
        inputs=inputs, targets=labels[picked], task="classification",
        ceiling=1.0, ceiling_metric="accuracy",
        description="XOR — not linearly separable",
    )


def make_spirals(n_samples: int = 512, n_turns: float = 2.0, noise_std: float = 0.1,
                 seed: int = 0) -> SyntheticDataset:
    """Two interleaved spirals. Ceiling: ~1.0 while the noise keeps them apart."""
    rng = np.random.default_rng(seed)
    half = n_samples // 2
    t = np.sqrt(rng.uniform(0, 1, half)) * n_turns * 2 * np.pi
    radius = t / (n_turns * 2 * np.pi) * 3

    def arm(phase):
        return np.stack([radius * np.cos(t + phase), radius * np.sin(t + phase)], axis=1)

    inputs = np.vstack([arm(0), arm(np.pi)]) + rng.normal(0, noise_std, size=(2 * half, 2))
    targets = np.concatenate([np.zeros(half), np.ones(half)])
    order = rng.permutation(len(inputs))
    return SyntheticDataset(
        inputs=inputs[order], targets=targets[order], task="classification",
        ceiling=1.0, ceiling_metric="accuracy",
        description=f"two spirals, {n_turns} turns, noise_std={noise_std}",
    )


def make_moons(n_samples: int = 512, separation: float = 0.5, noise_std: float = 0.1,
               seed: int = 0) -> SyntheticDataset:
    """Two interleaving half circles. Where k-means breaks (N02, N17)."""
    rng = np.random.default_rng(seed)
    half = n_samples // 2
    angle = rng.uniform(0, np.pi, half)
    upper = np.stack([np.cos(angle), np.sin(angle)], axis=1)
    lower = np.stack([1 - np.cos(angle), 1 - np.sin(angle) - separation], axis=1)
    inputs = np.vstack([upper, lower]) + rng.normal(0, noise_std, size=(2 * half, 2))
    targets = np.concatenate([np.zeros(half), np.ones(half)])
    order = rng.permutation(len(inputs))
    return SyntheticDataset(
        inputs=inputs[order], targets=targets[order], task="classification",
        ceiling=1.0, ceiling_metric="accuracy",
        description=f"two moons, separation={separation}",
    )


def make_pure_noise(n_samples: int = 512, n_features: int = 10,
                    seed: int = 0) -> SyntheticDataset:
    """Inputs and targets are INDEPENDENT. Ceiling: the majority class.

    ES: El dataset más importante del módulo. Cualquier modelo que supere el
    techo en validación está haciendo trampa o midiendo mal (N10, N14).
    """
    rng = np.random.default_rng(seed)
    inputs = rng.normal(size=(n_samples, n_features))
    targets = rng.integers(0, 2, size=n_samples)
    majority = max(np.mean(targets), 1 - np.mean(targets))
    return SyntheticDataset(
        inputs=inputs, targets=targets, task="classification",
        ceiling=float(majority), ceiling_metric="accuracy",
        description="pure noise — no relationship between inputs and targets",
    )


def make_shapes(n_samples: int = 512, image_size: int = 16, noise_std: float = 0.1,
                seed: int = 0) -> SyntheticDataset:
    """Tiny images of squares and circles, for CNNs without downloading anything."""
    rng = np.random.default_rng(seed)
    images = np.zeros((n_samples, 1, image_size, image_size))
    targets = rng.integers(0, 2, size=n_samples)
    grid_y, grid_x = np.mgrid[0:image_size, 0:image_size]

    for i, shape in enumerate(targets):
        size = rng.integers(4, image_size // 2)
        top = rng.integers(0, image_size - size)
        left = rng.integers(0, image_size - size)
        if shape == 0:
            images[i, 0, top:top + size, left:left + size] = 1.0
        else:
            center_y, center_x, radius = top + size / 2, left + size / 2, size / 2
            mask = (grid_y - center_y) ** 2 + (grid_x - center_x) ** 2 <= radius ** 2
            images[i, 0][mask] = 1.0

    images += rng.normal(0, noise_std, images.shape)
    dataset = SyntheticDataset(
        inputs=images.reshape(n_samples, -1), targets=targets, task="classification",
        ceiling=1.0, ceiling_metric="accuracy",
        description=f"squares vs circles, {image_size}x{image_size}",
        metadata={"image_size": image_size, "shape": (1, image_size, image_size)},
    )
    return dataset


# ─────────────────────────────────────────────────────────────────────────────
# Modifiers — everything we will want to break later is a parameter here
# ─────────────────────────────────────────────────────────────────────────────
def add_label_noise(dataset: SyntheticDataset, fraction: float,
                    seed: int = 0) -> SyntheticDataset:
    """Flip a fraction of labels. Lowers the ceiling to (1 - fraction).

    ES: El techo baja porque ni el modelo perfecto puede acertar una etiqueta
    que está mal. Ese descenso es lo que hace medible el ruido de etiqueta.
    """
    rng = np.random.default_rng(seed)
    corrupted = dataset._subset(np.arange(len(dataset)))
    n_flips = int(len(dataset) * fraction)
    flip_at = rng.choice(len(dataset), n_flips, replace=False)
    n_classes = dataset.n_classes
    corrupted.targets = corrupted.targets.copy()
    corrupted.targets[flip_at] = (corrupted.targets[flip_at] +
                                  rng.integers(1, n_classes, n_flips)) % n_classes
    corrupted.ceiling = dataset.ceiling * (1 - fraction)
    corrupted.description += f" + {fraction:.0%} label noise"
    return corrupted


def make_imbalanced(dataset: SyntheticDataset, minority_fraction: float,
                    seed: int = 0) -> SyntheticDataset:
    """Drop majority-class samples until the minority is `minority_fraction`."""
    rng = np.random.default_rng(seed)
    is_minority = dataset.targets == 1
    minority_idx = np.where(is_minority)[0]
    majority_idx = np.where(~is_minority)[0]
    n_minority = int(len(majority_idx) * minority_fraction / (1 - minority_fraction))
    keep = np.concatenate([majority_idx, rng.choice(minority_idx,
                                                    min(n_minority, len(minority_idx)),
                                                    replace=False)])
    rng.shuffle(keep)
    out = dataset._subset(keep)
    out.description += f" + imbalance ({minority_fraction:.0%} minority)"
    return out


def add_duplicates(dataset: SyntheticDataset, fraction: float,
                   seed: int = 0) -> SyntheticDataset:
    """Repeat a fraction of samples. Leaks across any random split (N15)."""
    rng = np.random.default_rng(seed)
    n_copies = int(len(dataset) * fraction)
    repeated = rng.choice(len(dataset), n_copies, replace=False)
    indices = np.concatenate([np.arange(len(dataset)), repeated])
    rng.shuffle(indices)
    out = dataset._subset(indices)
    out.description += f" + {fraction:.0%} duplicates"
    return out


def add_groups(dataset: SyntheticDataset, n_groups: int = 4, shift: float = 1.0,
               seed: int = 0) -> SyntheticDataset:
    """Assign samples to groups and shift each one. Random splits now leak (N15).

    ES: Simula sitios, sujetos o sensores distintos. El modelo puede aprender
    el grupo en vez del fenómeno, y el split aleatorio no lo detecta.
    """
    rng = np.random.default_rng(seed)
    groups = rng.integers(0, n_groups, size=len(dataset))
    offsets = rng.normal(0, shift, size=(n_groups, dataset.n_features))
    out = dataset._subset(np.arange(len(dataset)))
    out.inputs = (out.inputs + offsets[groups]).astype(np.float32)
    out.groups = groups
    out.description += f" + {n_groups} groups"
    return out


# ─────────────────────────────────────────────────────────────────────────────
# Harness registration — one thin wrapper per generator
# ─────────────────────────────────────────────────────────────────────────────
GENERATORS = {
    "line": make_line,
    "two_gaussians": make_two_gaussians,
    "xor": make_xor,
    "spirals": make_spirals,
    "moons": make_moons,
    "pure_noise": make_pure_noise,
    "shapes": make_shapes,
}


def _register_all() -> None:
    """Expose every generator to the harness as '<name>'.

    ES: Así el config dice {"dataset": "spirals"} y N01 no toca el arnés.
    """
    for name, generator in GENERATORS.items():
        def builder(_generator=generator, batch_size=32, val_fraction=0.2,
                    split_strategy="random", split_seed=0, **generator_kwargs):
            dataset = _generator(**generator_kwargs)
            return dataset.to_loaders(batch_size, val_fraction, split_strategy, split_seed)

        harness.datasets.register(name)(builder)


_register_all()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from lab import data
from lab import harness as H

for name, generator in data.GENERATORS.items():
    dataset = generator(n_samples=200)
    print(f"{name:16s} ceiling={dataset.ceiling:.4f} {dataset.ceiling_metric:9s} "
          f"features={dataset.n_features}")

## Verlos

Nada enseña más sobre un dataset que mirarlo antes de entrenar nada.

In [ ]:
two_dimensional = ["two_gaussians", "xor", "spirals", "moons"]
figure, axes = plt.subplots(1, 4, figsize=(15, 3.4))

for ax, name in zip(axes, two_dimensional):
    dataset = data.GENERATORS[name](n_samples=400)
    ax.scatter(dataset.inputs[:, 0], dataset.inputs[:, 1],
               c=dataset.targets, cmap="coolwarm", s=8, alpha=0.8)
    ax.set_title(f"{name}\nceiling {dataset.ceiling:.3f}", fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout(); plt.show()

## El techo, en detalle

Los tres casos que hay que distinguir, porque no todos los techos son iguales de fiables.

### ① Fórmula cerrada · dos gaussianas

Con dos gaussianas de igual covarianza separadas una distancia $d$, el error de Bayes es
exacto:

$$\text{error} = \Phi\!\left(-\frac{d}{2\sigma}\right)$$

Es el único generador del módulo donde el techo **no es una estimación**.

In [ ]:
from scipy.stats import norm

print("separación → techo (Bayes)")
for separation in [0.5, 1.0, 2.0, 4.0, 8.0]:
    dataset = data.make_two_gaussians(separation=separation, spread=1.0)
    formula = 1 - norm.cdf(-separation / 2)
    print(f"  d={separation:4.1f}   {dataset.ceiling:.4f}   (fórmula: {formula:.4f})")

### ② Derivado del ruido · regresión y ruido de etiqueta

En una regresión con ruido gaussiano, el mejor error cuadrático posible es $\sigma^2$: ni
el modelo perfecto puede predecir el ruido.

Con ruido de etiqueta pasa lo mismo en clasificación: si el $p$ % de las etiquetas está
mal, ningún modelo puede acertar más del $(1-p)$ %.

In [ ]:
clean = data.make_two_gaussians(n_samples=1000, separation=4.0)
print(f"limpio            ceiling = {clean.ceiling:.4f}")

for fraction in [0.05, 0.20, 0.40]:
    noisy = data.add_label_noise(clean, fraction=fraction, seed=0)
    print(f"{fraction:>5.0%} etiquetas mal   ceiling = {noisy.ceiling:.4f}   ← {noisy.description}")

### ③ El caso incómodo · ruido puro

Aquí no hay ninguna relación entre entradas y salidas. El techo es **la clase
mayoritaria**: lo mejor que puede hacer cualquier modelo es adivinar siempre la más
frecuente.

> ⚠️ **Cualquier cosa que supere ese techo en validación es un error de medición, no un
> descubrimiento.** Es el dataset que usaré en N10 y N14 precisamente por eso.

In [ ]:
noise_dataset = data.make_pure_noise(n_samples=1000, n_features=10, seed=0)
print(noise_dataset.summary())

class_counts = np.bincount(noise_dataset.targets)
print(f"\nreparto de clases: {class_counts}  →  mayoritaria = {class_counts.max()/len(noise_dataset):.3f}")

## 🔨 Qué rompo aquí · Aprender de lo que no existe

Entreno sobre ruido puro con un modelo con capacidad de sobra. Lo que espero:

- **Entrenamiento:** la pérdida baja y la exactitud sube. El modelo memoriza.
- **Validación:** se queda clavada en el azar.

Ese hueco entre las dos curvas es memorización pura, y verlo una vez vacuna para siempre.

In [ ]:
import torch
from torch import nn


@H.models.register("mlp")
def build_mlp(input_size, hidden_size=128, output_size=2, n_hidden_layers=2):
    layers, size = [], input_size
    for _ in range(n_hidden_layers):
        layers += [nn.Linear(size, hidden_size), nn.ReLU()]
        size = hidden_size
    layers.append(nn.Linear(size, output_size))
    return nn.Sequential(*layers)


@H.optimizers.register("adam")
def build_adam(params, **kwargs):
    return torch.optim.Adam(params, **kwargs)


noise_config = {
    "name": "n01-noise",
    "dataset": "pure_noise",
    "dataset_args": {"n_samples": 600, "n_features": 10, "batch_size": 32},
    "model": "mlp",
    "model_args": {"input_size": 10, "hidden_size": 128, "n_hidden_layers": 2},
    "optimizer": "adam",
    "optimizer_args": {"lr": 1e-3},
    "epochs": 120,
    "loss": "cross_entropy",
}

noise_result = H.run_experiment(noise_config, seed=0, verbose=False)
history = noise_result.history

plt.figure(figsize=(7, 4))
plt.plot([h["epoch"] for h in history], [h["train_loss"] for h in history], label="train")
plt.plot([h["epoch"] for h in history], [h["val_loss"] for h in history], label="validación")
plt.axhline(np.log(2), color="k", ls=":", lw=1, label="ln(2) — azar puro")
plt.xlabel("época"); plt.ylabel("pérdida"); plt.legend(); plt.grid(alpha=0.3)
plt.title("Ruido puro: el modelo memoriza lo que no existe")
plt.show()

print(f"train final: {history[-1]['train_loss']:.4f}")
print(f"val   final: {history[-1]['val_loss']:.4f}   (azar: {np.log(2):.4f})")

**Lectura.** La pérdida de entrenamiento se hunde: el modelo está memorizando 480 pares
`(entrada, etiqueta)` que no tienen ninguna relación. La de validación no solo no baja,
**empeora** — porque el modelo aplica con confianza un patrón inventado.

Si esto lo hubiera visto sin conocer el techo, podría haber pensado que el modelo
"aprende algo pero generaliza mal". Con el techo delante, la lectura es inequívoca: aquí
no hay nada que aprender.

> Esta es la razón de la decisión de diseño número 1. Vuelve a ella en N10 y N14.

---
---

# Parte B · El lenguaje aritmético

Lo que atraviesa de N18 a N30. Diecisiete tokens y un verificador de tres líneas.

In [ ]:
%%writefile lab/arithmetic.py
"""The arithmetic language: the thread that runs from N18 to N30.

ES: El lenguaje aritmético. Se eligió porque cumple cuatro cosas a la vez:
verificador de tres líneas, preferencias objetivas sin anotar, dificultad
graduable, y todo cabe en CPU.

Comment convention / Convenio: docstrings in English say WHAT, Spanish notes say WHY.
"""
from __future__ import annotations

import ast
import operator
import random
from dataclasses import dataclass
from typing import Literal

Level = Literal[0, 1, 2, 3]
FormatStyle = Literal["raw", "chat", "reversed"]
SplitStrategy = Literal["random", "by_result", "by_range"]
PreferenceKind = Literal["correctness", "brevity", "honesty"]

USER_TOKEN = "<|user|>"
ASSISTANT_TOKEN = "<|assistant|>"
END_TOKEN = "<|end|>"
PAD_TOKEN = "<|pad|>"
SPECIAL_TOKENS = [PAD_TOKEN, END_TOKEN, USER_TOKEN, ASSISTANT_TOKEN]

DIGITS = list("0123456789")
SYMBOLS = list("+-*()=")
VOCABULARY = SPECIAL_TOKENS + DIGITS + SYMBOLS

TOKEN_TO_ID = {token: index for index, token in enumerate(VOCABULARY)}
ID_TO_TOKEN = {index: token for token, index in TOKEN_TO_ID.items()}

_OPERATIONS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul}

QUESTION_TEMPLATES = [
    "¿cuánto es {expression}?",
    "calcula {expression}",
    "{expression} = ?",
    "resuelve {expression}",
]


# ─────────────────────────────────────────────────────────────────────────────
# Solving and verifying
# ─────────────────────────────────────────────────────────────────────────────
def solve(expression: str) -> str:
    """Evaluate an expression safely. This is also the reward signal for N28.

    ES: Se usa `ast` en vez de `eval` porque un verificador que ejecuta código
    arbitrario deja de ser un verificador.
    """
    return str(_evaluate(ast.parse(expression, mode="eval").body))


def _evaluate(node: ast.AST) -> int:
    if isinstance(node, ast.Constant):
        return node.value
    if isinstance(node, ast.BinOp):
        return _OPERATIONS[type(node.op)](_evaluate(node.left), _evaluate(node.right))
    if isinstance(node, ast.UnaryOp) and isinstance(node.op, ast.USub):
        return -_evaluate(node.operand)
    raise ValueError(f"unsupported expression node: {ast.dump(node)}")


def verify(expression: str, answer: str) -> bool:
    """Is this answer correct? The whole reward function for RLVR fits here."""
    try:
        return answer.strip() == solve(expression)
    except Exception:
        return False


def reasoning_steps(expression: str) -> str:
    """Intermediate steps, as a reference trace for N29.

    ES: En N28 el modelo debería DESCUBRIR algo parecido por su cuenta. Esto
    es la respuesta que le habríamos enseñado, para poder comparar.
    """
    steps: list[str] = []
    _collect_steps(ast.parse(expression, mode="eval").body, steps)
    return ", ".join(steps)


def _collect_steps(node: ast.AST, steps: list[str]) -> int:
    if isinstance(node, ast.Constant):
        return node.value
    left = _collect_steps(node.left, steps)
    right = _collect_steps(node.right, steps)
    symbol = {ast.Add: "+", ast.Sub: "-", ast.Mult: "*"}[type(node.op)]
    result = _OPERATIONS[type(node.op)](left, right)
    steps.append(f"{left}{symbol}{right}={result}")
    return result


# ─────────────────────────────────────────────────────────────────────────────
# Generation
# ─────────────────────────────────────────────────────────────────────────────
@dataclass(frozen=True)
class Problem:
    """An expression and its answer. Everything else is derived."""

    expression: str
    answer: str
    level: int

    @property
    def raw(self) -> str:
        return f"{self.expression}={self.answer}"

    def __str__(self) -> str:
        return self.raw


def generate(level: Level = 1, n_problems: int = 1000, seed: int = 0,
             max_digits: int | None = None,
             allow_negative: bool = False) -> list[Problem]:
    """Build n problems of the requested difficulty level.

    ES: El nivel es el único mando de dificultad. Subes de nivel cuando el
    anterior se resuelve, no antes.

    `allow_negative=False` mantiene el problema pequeño: sin resultados
    negativos, el modelo no tiene que aprender el signo además de la
    aritmética. Actívalo cuando quieras subir la dificultad sin cambiar de
    nivel.
    """
    rng = random.Random(seed)
    builders = {0: _level_0, 1: _level_1, 2: _level_2, 3: _level_3}
    if level not in builders:
        raise ValueError(f"level must be one of {sorted(builders)}")

    digits = max_digits or {0: 1, 1: 3, 2: 2, 3: 2}[level]
    problems = []
    while len(problems) < n_problems:
        expression = builders[level](rng, digits)
        answer = solve(expression)
        if not allow_negative and answer.startswith("-"):
            continue
        problems.append(Problem(expression, answer, level))
    return problems


def _number(rng: random.Random, max_digits: int) -> int:
    return rng.randint(0, 10 ** rng.randint(1, max_digits) - 1)


def _level_0(rng: random.Random, _max_digits: int) -> str:
    return f"{rng.randint(0, 9)}+{rng.randint(0, 9)}"


def _level_1(rng: random.Random, max_digits: int) -> str:
    return f"{_number(rng, max_digits)}+{_number(rng, max_digits)}"


def _level_2(rng: random.Random, max_digits: int) -> str:
    """Three operands, no parentheses: the model must learn precedence."""
    a, b, c = (_number(rng, max_digits) for _ in range(3))
    return f"{a}{rng.choice('+-*')}{b}{rng.choice('+-*')}{c}"


def _level_3(rng: random.Random, max_digits: int) -> str:
    """Parentheses: hard in one pass, easy step by step. That gap is what makes
    the reasoning emerge in N28."""
    a, b, c = (_number(rng, max_digits) for _ in range(3))
    inner, outer = rng.choice("+-"), rng.choice("+-*")
    if rng.random() < 0.5:
        return f"({a}{inner}{b}){outer}{c}"
    return f"{a}{outer}({b}{inner}{c})"


# ─────────────────────────────────────────────────────────────────────────────
# Formatting
# ─────────────────────────────────────────────────────────────────────────────
def format_example(problem: Problem, style: FormatStyle = "raw",
                   seed: int | None = None) -> str:
    """Render a problem in one of the three training formats.

    ES: 'reversed' existe para un experimento concreto de N18: con la respuesta
    al revés el acarreo fluye en el sentido de la generación, y el modelo lo
    aprende mucho antes. La representación del dato importa tanto como la
    arquitectura.
    """
    if style == "raw":
        return problem.raw
    if style == "reversed":
        return f"{problem.expression}={problem.answer[::-1]}"
    if style == "chat":
        rng = random.Random(seed if seed is not None else hash(problem.expression))
        question = rng.choice(QUESTION_TEMPLATES).format(expression=problem.expression)
        return f"{USER_TOKEN}{question}{ASSISTANT_TOKEN}{problem.answer}{END_TOKEN}"
    raise ValueError(f"unknown style '{style}'")


def build_corpus(problems: list[Problem], style: FormatStyle = "raw",
                 separator: str = "\n") -> str:
    """One long string, ready for next-token pretraining (N21)."""
    return separator.join(format_example(p, style) for p in problems)


# ─────────────────────────────────────────────────────────────────────────────
# Tokenizing
# ─────────────────────────────────────────────────────────────────────────────
def tokenize(text: str, group_digits: bool = False) -> list[str]:
    """Split text into tokens. Character level by default.

    ES: `group_digits=True` existe solo para demostrar que empeora la
    aritmética. Es la razón por la que los modelos reales fallan al sumar.
    """
    tokens, position = [], 0
    while position < len(text):
        special = next((s for s in SPECIAL_TOKENS if text.startswith(s, position)), None)
        if special:
            tokens.append(special)
            position += len(special)
            continue

        character = text[position]
        if group_digits and character.isdigit():
            end = position
            while end < len(text) and text[end].isdigit():
                end += 1
            tokens.append(text[position:end])
            position = end
            continue

        tokens.append(character)
        position += 1
    return tokens


def encode(text: str) -> list[int]:
    """Token ids, character level. Unknown characters are skipped."""
    return [TOKEN_TO_ID[t] for t in tokenize(text) if t in TOKEN_TO_ID]


def decode(token_ids: list[int]) -> str:
    return "".join(ID_TO_TOKEN[i] for i in token_ids)


# ─────────────────────────────────────────────────────────────────────────────
# Preference pairs — objective, no human annotation needed
# ─────────────────────────────────────────────────────────────────────────────
@dataclass(frozen=True)
class PreferencePair:
    """A prompt and two answers, one preferred over the other."""

    prompt: str
    preferred: str
    rejected: str
    kind: str


def generate_preferences(kind: PreferenceKind = "correctness", n_pairs: int = 500,
                         level: Level = 2, seed: int = 0,
                         out_of_range_digits: int = 6) -> list[PreferencePair]:
    """Build preference pairs without a single human annotation.

    ES: Esta función es la razón principal de haber elegido aritmética. En un
    dominio real, estos pares costarían semanas de anotadores.
    """
    rng = random.Random(seed)
    builders = {"correctness": _pairs_correctness,
                "brevity": _pairs_brevity,
                "honesty": _pairs_honesty}
    if kind not in builders:
        raise ValueError(f"kind must be one of {sorted(builders)}")
    return builders[kind](rng, n_pairs, level, out_of_range_digits)


def _pairs_correctness(rng, n_pairs, level, _digits):
    pairs = []
    for problem in generate(level, n_pairs, seed=rng.randint(0, 10 ** 6)):
        wrong = str(int(problem.answer) + rng.choice([-2, -1, 1, 2]))
        pairs.append(PreferencePair(problem.expression, problem.answer, wrong, "correctness"))
    return pairs


def _pairs_brevity(rng, n_pairs, level, _digits):
    padding = ["El resultado de la operación {e} es, efectivamente, {a}.",
               "Vamos a calcularlo con calma. Tenemos {e}, y el resultado final es {a}.",
               "Para responder a {e}, hay que operar paso a paso; el resultado es {a}."]
    pairs = []
    for problem in generate(level, n_pairs, seed=rng.randint(0, 10 ** 6)):
        inflated = rng.choice(padding).format(e=problem.expression, a=problem.answer)
        pairs.append(PreferencePair(problem.expression, problem.answer, inflated, "brevity"))
    return pairs


def _pairs_honesty(rng, n_pairs, level, out_of_range_digits):
    """Numbers far outside the training range: the model CANNOT know the answer.

    ES: Y por eso se puede medir si aprendió a decir "no lo sé", cosa que con
    datos reales es casi imposible de comprobar.
    """
    pairs = []
    lower = 10 ** (out_of_range_digits - 1)
    upper = 10 ** out_of_range_digits - 1
    for _ in range(n_pairs):
        expression = f"{rng.randint(lower, upper)}+{rng.randint(lower, upper)}"
        invented = str(rng.randint(lower, upper * 2))
        pairs.append(PreferencePair(expression, "no lo sé", invented, "honesty"))
    return pairs


# ─────────────────────────────────────────────────────────────────────────────
# Splitting — three strategies that measure three different things
# ─────────────────────────────────────────────────────────────────────────────
def split_problems(problems: list[Problem], strategy: SplitStrategy = "random",
                   val_fraction: float = 0.2,
                   seed: int = 0) -> tuple[list[Problem], list[Problem]]:
    """Split problems into train and validation.

    ES: Los tres miden cosas distintas, y ahí está la trampa de este dataset:
      random     → 47+38 en train y 38+47 en validación. ¿Generalizó o
                   memorizó la conmutatividad?
      by_result  → todos los que dan 85 caen del mismo lado. Mucho más duro.
      by_range   → entrena con pocos dígitos, evalúa con más. Extrapolación.
    """
    rng = random.Random(seed)
    shuffled = problems[:]
    rng.shuffle(shuffled)

    if strategy == "random":
        cut = int(len(shuffled) * (1 - val_fraction))
        return shuffled[:cut], shuffled[cut:]

    if strategy == "by_result":
        results = sorted({p.answer for p in shuffled})
        rng.shuffle(results)
        n_val = max(1, int(len(results) * val_fraction))
        val_results = set(results[:n_val])
        train = [p for p in shuffled if p.answer not in val_results]
        val = [p for p in shuffled if p.answer in val_results]
        return train, val

    if strategy == "by_range":
        def longest_operand(problem: Problem) -> int:
            numbers = [n for n in _split_numbers(problem.expression)]
            return max(len(n) for n in numbers)

        lengths = sorted({longest_operand(p) for p in shuffled})
        threshold = lengths[-1] if len(lengths) > 1 else lengths[0]
        train = [p for p in shuffled if longest_operand(p) < threshold]
        val = [p for p in shuffled if longest_operand(p) >= threshold]
        return train, val

    raise ValueError(f"unknown strategy '{strategy}'")


def _split_numbers(expression: str) -> list[str]:
    numbers, current = [], ""
    for character in expression:
        if character.isdigit():
            current += character
        elif current:
            numbers.append(current)
            current = ""
    if current:
        numbers.append(current)
    return numbers

In [ ]:
from lab import arithmetic as A

print(f"vocabulario: {len(A.VOCABULARY)} tokens")
print(A.VOCABULARY)

## Los cuatro niveles

Un único parámetro controla la dificultad. Se sube de nivel cuando el anterior se
resuelve, no antes.

In [ ]:
for level in [0, 1, 2, 3]:
    examples = A.generate(level=level, n_problems=5, seed=level)
    print(f"nivel {level}:  " + "   ".join(p.raw for p in examples))

| Nivel | Qué añade | Se usa en |
|---|---|---|
| **0** | Un dígito. Solo 100 combinaciones: se puede memorizar | N16–N18 |
| **1** | Dos y tres dígitos → **acarreo**, la dependencia de largo alcance | N18–N21 |
| **2** | Varios operadores → hay que decidir el **orden** | N21–N25 |
| **3** | Paréntesis → difícil de una pasada, fácil por pasos | **N26–N30** |

El nivel 3 es el importante: esa brecha entre "difícil de una pasada" y "fácil por pasos"
es exactamente lo que hace que el razonamiento **emerja solo** en N28.

## El verificador

Tres líneas. Y es lo que hace posible todo el bloque 6.

In [ ]:
problem = A.generate(level=3, n_problems=1, seed=7)[0]

print(f"problema : {problem.expression}")
print(f"respuesta: {problem.answer}")
print(f"pasos    : {A.reasoning_steps(problem.expression)}")
print()
print(f"verify(expr, '{problem.answer}')  → {A.verify(problem.expression, problem.answer)}")
print(f"verify(expr, '999')  → {A.verify(problem.expression, '999')}")

La línea de **pasos** es la traza de referencia: lo que le habríamos enseñado al modelo.
En N28 la pregunta es si la descubre por su cuenta, entrenando solo con la recompensa
del verificador.

## Los tres formatos

Cada uno alimenta una fase distinta del pipeline.

In [ ]:
sample = A.generate(level=1, n_problems=1, seed=3)[0]

print("raw       ", A.format_example(sample, "raw"), "     ← preentrenamiento (N21)")
print("chat      ", A.format_example(sample, "chat"), "  ← SFT (N25)")
print("reversed  ", A.format_example(sample, "reversed"), "     ← el experimento de N18")

### Por qué existe `reversed`

Con la respuesta escrita al revés, el **acarreo fluye en el mismo sentido que la
generación**: el modelo produce primero las unidades, que es donde empieza el acarreo.

Es mucho más fácil de aprender, y es un experimento de diez minutos que demuestra algo
que no es obvio: **la representación del dato importa tanto como la arquitectura**.

## Tokenización, y por qué los modelos reales fallan sumando

In [ ]:
text = "1234+5678=6912"

print("carácter :", A.tokenize(text))
print("agrupado :", A.tokenize(text, group_digits=True))
print()
print("ids      :", A.encode("47+38=85"))
print("decode   :", A.decode(A.encode("47+38=85")))

El modo agrupado es más corto y **peor para la aritmética**: el modelo nunca ve los
dígitos por separado, así que no puede alinear unidades con unidades.

> Los modelos reales tokenizan números de forma inconsistente —`1234` puede ser uno, dos
> o cuatro tokens según el contexto— y esa es una de las causas de que fallen en
> operaciones simples. Aquí está el mando para comprobarlo en N21.

## Preferencias sin anotar a nadie

**La razón principal de haber elegido aritmética.** En un dominio real, estos pares
costarían semanas de anotadores.

In [ ]:
for kind in ["correctness", "brevity", "honesty"]:
    pair = A.generate_preferences(kind=kind, n_pairs=1, level=2, seed=5)[0]
    print(f"── {kind}")
    print(f"   prompt   : {pair.prompt}")
    print(f"   preferida: {pair.preferred!r}")
    print(f"   rechazada: {pair.rejected[:60]!r}")
    print()

El tercero es el más interesante y solo es posible aquí: los números están **fuera del
rango de entrenamiento**, así que el modelo *no puede* saber la respuesta.

Si prefiere "no lo sé" a una invención, le estamos enseñando a admitir ignorancia — y,
a diferencia de lo que pasa con datos reales, **podemos medir si lo aprendió**.

## 🔨 Qué rompo aquí (2) · La trampa del split aleatorio

Con aritmética hay una trampa preciosa, y es la lección de → 3.9 sobre un caso donde se
ve.

Si parto al azar, `47+38` puede quedar en entrenamiento y `38+47` en validación. Si el
modelo acierta, **¿generalizó o memorizó la conmutatividad?** No hay forma de saberlo.

Los tres splits miden cosas distintas.

In [ ]:
problems = A.generate(level=1, n_problems=2000, seed=0)

for strategy in ["random", "by_result", "by_range"]:
    train, val = A.split_problems(problems, strategy=strategy, seed=0)
    train_answers = {p.answer for p in train}
    leaked = sum(1 for p in val if p.answer in train_answers) / max(len(val), 1)
    print(f"{strategy:10s}  train={len(train):4d}  val={len(val):4d}  "
          f"respuestas de val ya vistas en train: {leaked:.0%}")

**Lectura de la tabla.**

- `random` — casi todas las respuestas de validación aparecen ya en entrenamiento. El
  modelo puede acertar recordando el resultado, no calculándolo.
- `by_result` — cero solapamiento. Mucho más duro, y mucho más informativo.
- `by_range` — entrena con pocos dígitos y evalúa con más. Mide **extrapolación**, que es
  otra cosa distinta.

> **Ninguno es "el correcto".** Cada uno responde a una pregunta diferente, y elegir mal
> el split es la forma más silenciosa de inflar un resultado (→ N15).

---

## Comprobación final

Con esto, ningún notebook posterior necesita escribir código de datos.

In [ ]:
checks = {
    "generadores tabulares registrados": all(n in H.datasets for n in data.GENERATORS),
    "todos traen techo":                 all(g(n_samples=50).ceiling is not None
                                             for g in data.GENERATORS.values()),
    "modificadores disponibles":         all(hasattr(data, f) for f in
                                             ["add_label_noise", "make_imbalanced",
                                              "add_duplicates", "add_groups"]),
    "cuatro niveles generan":            all(len(A.generate(l, 5, seed=0)) == 5
                                             for l in [0, 1, 2, 3]),
    "el verificador funciona":           A.verify("(3+4)*2", "14") and not A.verify("(3+4)*2", "13"),
    "tres formatos":                     all(A.format_example(A.generate(1,1,seed=0)[0], s)
                                             for s in ["raw", "chat", "reversed"]),
    "tres tipos de preferencia":         all(A.generate_preferences(k, 2, seed=0)
                                             for k in ["correctness", "brevity", "honesty"]),
    "tres estrategias de split":         all(A.split_problems(A.generate(1, 100, seed=0), s)
                                             for s in ["random", "by_result", "by_range"]),
}

for label, passed in checks.items():
    print(f"{'✓' if passed else '✗'} {label}")

assert all(checks.values()), "algo no está listo para N02"

---

## Criterio de terminado

- [x] Siete generadores tabulares, todos con **techo** y semilla
- [x] Cinco modificadores para provocar fallos después, sin reescribir nada
- [x] El lenguaje aritmético con cuatro niveles y verificador
- [x] Tres formatos, tres tipos de preferencia, tres estrategias de split
- [x] Todo registrado en el arnés **sin tocar `harness.py`**
- [x] He visto a un modelo memorizar ruido puro

---

## Cierre de bitácora

*(Copiar a la entrada de bitácora antes de cerrar el notebook.)*

### Qué aprendí

### Qué me sorprendió

> Candidatos, si no se te ocurre nada: cuánto solapamiento tiene un split aleatorio en
> aritmética, o lo rápido que memoriza el ruido puro un modelo mediano.

### Qué haría distinto

### Siguiente paso

**N02 · El perceptrón, y su muerte.** Usa `make_xor()` de este notebook, y su techo es
1.0 — perfectamente alcanzable, y ninguna recta puede llegar.